# Notebook 3: Hallucination detection with retrieved context

This notebook runs two small language models (Qwen3-4B and Phi-4-mini) on ARC-Challenge science questions **with retrieved passages added to the prompt**. It then measures how well two detectors (D0: answer agreement, D1: explanation similarity) separate wrong answers from correct ones, and compares the results with the no-context run from Notebook 1.



## 1. Install packages
Installs the libraries needed for loading models, datasets and data handling.

In [ ]:
!pip -q install pandas==2.2.3 transformers accelerate datasets

## 2. Imports, Hugging Face login and Google Drive
Imports the main libraries, logs in to Hugging Face with the token stored in Colab Secrets (`HF_TOKEN`), mounts Google Drive and creates the project folder `hallu_project`. It also prints whether a GPU is available.

In [ ]:
import json, os, re, random, time, gc, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, transformers
from google.colab import drive, userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))
drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/hallu_project")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print("GPU:", torch.cuda.is_available(), "|", sorted(p.name for p in PROJECT_DIR.glob("*")))

Mounted at /content/drive
GPU: False | ['data', 'gen_phi4_mini_ctx_dev.jsonl', 'gen_phi4_mini_ctx_test.jsonl', 'gen_phi4_mini_no_ctx_dev.jsonl', 'gen_phi4_mini_no_ctx_test.jsonl', 'gen_qwen3_4b_ctx_dev.jsonl', 'gen_qwen3_4b_ctx_test.jsonl', 'gen_qwen3_4b_no_ctx_dev.jsonl', 'gen_qwen3_4b_no_ctx_test.jsonl', 'pilot_phi4_mini.csv', 'pilot_qwen3_4b.csv', 'question_sets.json', 'questions_resolved.json', 'retrieval_hits_dev.json', 'retrieval_hits_test.json', 'retrieved_dev.json', 'retrieved_test.json', 'run_info.jsonl', 'scores_phi4_mini_ctx_dev.csv', 'scores_phi4_mini_ctx_test.csv', 'scores_phi4_mini_no_ctx_dev.csv', 'scores_phi4_mini_no_ctx_test.csv', 'scores_qwen3_4b_ctx_dev.csv', 'scores_qwen3_4b_ctx_test.csv', 'scores_qwen3_4b_no_ctx_dev.csv', 'scores_qwen3_4b_no_ctx_test.csv']


## 3. Load ARC-Challenge and fixed question sets
Loads the ARC-Challenge dataset and defines how a question is formatted (options A to E) and how the answer letter is read from the model output. It loads the fixed dev (150) and test (300) question ids, or creates them once with seed 42, and saves the environment details to `run_info.jsonl`.

In [ ]:
from datasets import load_dataset
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge")

def format_q(ex):
    labels, texts = ex["choices"]["label"], ex["choices"]["text"]
    letters = "ABCDE"[:len(labels)]
    opts = "\n".join(f"{l}. {t}" for l, t in zip(letters, texts))
    gold = letters[labels.index(ex["answerKey"])]
    return ex["question"], opts, gold, letters

def parse_letter(text, letters):
    m = re.search(r"Answer:\s*\(?([A-E])\b", text)
    return m.group(1) if m and m.group(1) in letters else None

sets_path = PROJECT_DIR / "question_sets.json"
if sets_path.exists():
    sets = json.load(open(sets_path))
else:
    rng = random.Random(42)
    dev_idx = rng.sample(range(len(arc["validation"])), 150)
    test_idx = rng.sample(range(len(arc["test"])), 300)
    sets = {"dev": [arc["validation"][i]["id"] for i in dev_idx],
            "test": [arc["test"][i]["id"] for i in test_idx]}
    json.dump(sets, open(sets_path, "w"))
by_id = {ex["id"]: ex for s in ("validation", "test") for ex in arc[s]}
print("dev:", len(sets["dev"]), "| test:", len(sets["test"]))

info = dict(date=time.strftime("%Y-%m-%d %H:%M"), python=platform.python_version(),
            torch=torch.__version__, transformers=transformers.__version__,
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
with open(PROJECT_DIR / "run_info.jsonl", "a") as f:
    f.write(json.dumps(info) + "\n")
print(info)

README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  190kB            

ARC-Challenge/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

ARC-Challenge/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B /  204kB            

ARC-Challenge/test-00000-of-00001.parque(…): downloading bytes:           |  0.00B            

ARC-Challenge/validation-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 55.7kB            

ARC-Challenge/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

dev: 150 | test: 300
{'date': '2026-11-22 06:16', 'python': '3.13.18', 'torch': '2.14.1+cpu', 'transformers': '5.22.1', 'gpu': 'none'}


## 4. Model and generation helpers
Defines how to load a model, build the prompt (with an optional context), get one main answer (greedy) and 5 sampled answers (temperature 0.7). `run_condition` runs a whole split and saves every result to a `.jsonl` file in Drive. If the run stops, running it again skips the questions that are already done.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="auto")
    return tok, model

def build_prompt(question, opts, context=None):
    ctx = f"Context:\n{context}\n\n" if context else ""
    return ("Answer the science question. Reply in exactly this format:\n"
            "Answer: <option letter>\nReason: <one sentence>\n\n"
            f"{ctx}Question: {question}\n{opts}")

def _inputs(tok, model, question, opts, context):
    text = tok.apply_chat_template(
        [{"role": "user", "content": build_prompt(question, opts, context)}],
        tokenize=False, add_generation_prompt=True)
    return tok(text, return_tensors="pt").to(model.device)

def _decode(tok, out, n_in):
    return [tok.decode(o[n_in:], skip_special_tokens=True) for o in out]

def ask(tok, model, question, opts, context=None, max_new_tokens=60):
    inp = _inputs(tok, model, question, opts, context)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return _decode(tok, out, inp["input_ids"].shape[1])[0]

def ask_many(tok, model, question, opts, context=None, n=5, temperature=0.7, max_new_tokens=60):
    inp = _inputs(tok, model, question, opts, context)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=temperature, top_p=1.0, num_return_sequences=n)
    return _decode(tok, out, inp["input_ids"].shape[1])

def run_condition(tok, model, tag, split, condition="no_ctx", contexts=None):
    path = PROJECT_DIR / f"gen_{tag}_{condition}_{split}.jsonl"
    done = set()
    if path.exists():
        with open(path) as f:
            done = {json.loads(l)["id"] for l in f}
    ids = [i for i in sets[split] if i not in done]
    print(f"{tag} {condition} {split}: {len(done)} done, {len(ids)} to go")
    t_start = time.time()
    with open(path, "a") as f:
        for k, qid in enumerate(ids, 1):
            ex = by_id[qid]
            q, opts, gold, letters = format_q(ex)
            ctx = contexts.get(qid) if contexts else None
            t0 = time.time(); main = ask(tok, model, q, opts, ctx); t1 = time.time()
            samples = ask_many(tok, model, q, opts, ctx); t2 = time.time()
            pred = parse_letter(main, letters)
            rec = dict(id=qid, split=split, condition=condition, model=tag, gold=gold,
                       main=main, pred=pred, correct=(pred == gold),
                       samples=samples, sample_preds=[parse_letter(s, letters) for s in samples],
                       sec_main=round(t1 - t0, 2), sec_samples=round(t2 - t1, 2))
            f.write(json.dumps(rec) + "\n"); f.flush()
            if k % 25 == 0:
                print(f"  {k}/{len(ids)} | {round((time.time()-t_start)/60,1)} min")
    print("finished", path.name)

## 5. Analysis helpers (D0 and AUROC)
Functions to load saved generations, compute answer agreement (the basis of D0), calculate bootstrap confidence intervals for AUROC, and print a summary of accuracy and D0 AUROC. D0 score = 1 minus the share of the 5 samples that match the main answer.

In [ ]:
from sklearn.metrics import roc_auc_score

def load_gen(tag, condition, split):
    path = PROJECT_DIR / f"gen_{tag}_{condition}_{split}.jsonl"
    return pd.DataFrame([json.loads(l) for l in open(path)])

def agreement(r):
    return np.mean([p == r.pred for p in r.sample_preds])

def boot_auc(y, s, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    y, s = np.asarray(y), np.asarray(s)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if y[idx].min() == y[idx].max():
            continue
        vals.append(roc_auc_score(y[idx], s[idx]))
    return np.percentile(vals, [2.5, 97.5])

def summarize(tag, condition, split):
    d = load_gen(tag, condition, split)
    err = (~d.correct).values
    score = 1 - d.apply(agreement, axis=1).values
    wrong = d[err]
    print(f"{tag} | {condition} | {split}: n={len(d)}, errors={err.sum()}, accuracy={d.correct.mean():.3f}")
    print("  main unparsed:", int(d.pred.isna().sum()),
          "| all-5-agree:", round(d.sample_preds.apply(lambda ps: len(set(ps)) == 1).mean(), 3))
    print("  wrong & all 5 repeat same wrong option:",
          round(wrong.apply(lambda r: all(p == r.pred for p in r.sample_preds), axis=1).mean(), 3))
    auc = roc_auc_score(err, score); lo, hi = boot_auc(err, score)
    print(f"  D0 AUROC: {auc:.3f}  (95% CI {lo:.3f} to {hi:.3f})")
    return d

## 6. Load the sentence embedding model
Loads all-MiniLM-L6-v2, which turns each explanation into a vector so that explanations can be compared. It is used for the D1 detector.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    !pip -q install sentence-transformers
    from sentence_transformers import SentenceTransformer

emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("embedding model ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding model ready


## 7. D1 detector and evaluation function
Extracts the one-sentence reason from each answer, measures how similar the main reason is to the 5 sampled reasons, and defines D1 as 1 minus the average similarity. `eval_d1` computes D0 and D1 AUROC with confidence intervals and saves the scores to a CSV file.

In [ ]:
def reason(txt):
    m = re.search(r"Reason:\s*(.*)", txt, re.S)
    return (m.group(1) if m else txt).strip()

def d1_score(r):
    texts = [reason(r.main)] + [reason(s) for s in r.samples]
    E = emb.encode(texts, normalize_embeddings=True)
    return 1 - float(np.mean(E[1:] @ E[0]))

def eval_d1(tag, condition, split):
    d = load_gen(tag, condition, split)
    err = (~d.correct).values
    d0 = 1 - d.apply(agreement, axis=1).values
    d1 = d.apply(d1_score, axis=1).values
    out = pd.DataFrame({"id": d.id, "error": err, "d0": d0, "d1": d1})
    out.to_csv(PROJECT_DIR / f"scores_{tag}_{condition}_{split}.csv", index=False)
    a0, a1 = roc_auc_score(err, d0), roc_auc_score(err, d1)
    l0, h0 = boot_auc(err, d0); l1, h1 = boot_auc(err, d1)
    print(f"{tag} | {condition} | {split}: n={len(d)}, errors={err.sum()}")
    print(f"  D0 AUROC {a0:.3f} (95% CI {l0:.3f}-{h0:.3f}) | D1 AUROC {a1:.3f} (95% CI {l1:.3f}-{h1:.3f})")
    print(f"  D1 mean score: correct={d1[~err].mean():.4f} | wrong={d1[err].mean():.4f}")
    silent = (d0 == 0)
    if err[silent].sum() > 0 and (~err[silent]).sum() > 0:
        print(f"  where D0 is silent (all 5 agree, n={silent.sum()}): errors={err[silent].sum()}, D1 AUROC {roc_auc_score(err[silent], d1[silent]):.3f}")
    return out

## 8. Check the format of the retrieved passages
Opens the retrieved passage files from Notebook 2 and prints one example from each, to confirm how the context is stored.

In [ ]:
for split in ["dev", "test"]:
    r = json.load(open(PROJECT_DIR / f"retrieved_{split}.json"))
    print(split, type(r).__name__, len(r))
    first = r[0] if isinstance(r, list) else r[next(iter(r))]
    print(json.dumps(first, indent=1)[:800])
    print("-" * 40)

dev dict 150
"- Cell wall, cell membrane, chloroplasts, nucleus, and vacuole.\n- Look for cytoplasm, nucleus, chloroplasts, cell wall, and cell membrane.\n- Which of the following is a function of the cell membrane?"
----------------------------------------
test dict 300
"- Ocean tides are caused by the gravity of the moon.\n- The Moon causes these tides by deforming the oceans, and as the Earth rotates under this ocean bulge, it causes a high tide to propagate onto beaches.\n- Earth's tides are caused by the moon's gravitational pull on the oceans."
----------------------------------------


## 9. Build the CONTEXTS dictionary
Turns the retrieved passages into context text for each question id (`CONTEXTS["dev"]` and `CONTEXTS["test"]`). It also prints how many contexts were created, their average length, and how many questions have no context (should be 0).

In [ ]:
def to_ctx(v):
    if isinstance(v, str):
        return v.strip()
    parts = []
    for p in v:
        if isinstance(p, dict):
            p = p.get("text") or p.get("passage") or p.get("doc") or ""
        parts.append(str(p).strip())
    return "\n".join(f"[{i+1}] {p}" for i, p in enumerate(parts) if p)

CONTEXTS = {}
for split in ["dev", "test"]:
    r = json.load(open(PROJECT_DIR / f"retrieved_{split}.json"))
    if isinstance(r, list):
        r = {x["id"]: x.get("passages") or x.get("retrieved") or x.get("contexts") for x in r}
    CONTEXTS[split] = {qid: to_ctx(v) for qid, v in r.items()}
    lens = [len(c) for c in CONTEXTS[split].values()]
    missing = [q for q in sets[split] if not CONTEXTS[split].get(q)]
    print(f"{split}: {len(CONTEXTS[split])} contexts | avg chars={np.mean(lens):.0f}, max={max(lens)} | questions without context: {len(missing)}")

dev: 150 contexts | avg chars=260, max=486 | questions without context: 0
test: 300 contexts | avg chars=246, max=494 | questions without context: 0


## 10. Load a model and generate with context on the dev set
Loads the model chosen in the `tag` line (change only that line to switch between `qwen3_4b` and `phi4_mini`), frees the previous model from GPU memory, and generates answers for the 150 dev questions with the retrieved context.

In [ ]:
MODELS = {"qwen3_4b":  "Qwen/Qwen3-4B-Instruct-2507",
          "phi4_mini": "microsoft/Phi-4-mini-instruct"}

tag = "qwen3_4b"        # <- To change the model make change to this line only ("qwen3_4b" ya "phi4_mini")

if "model" in globals():
    del model; gc.collect(); torch.cuda.empty_cache()

tok, model = load_model(MODELS[tag])
run_condition(tok, model, tag, "dev", condition="ctx", contexts=CONTEXTS["dev"])

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

qwen3_4b ctx dev: 0 done, 150 to go
  25/150 | 1.8 min
  50/150 | 3.6 min
  75/150 | 5.4 min
  100/150 | 7.3 min
  125/150 | 9.1 min
  150/150 | 10.8 min
finished gen_qwen3_4b_ctx_dev.jsonl


## 11. Generate with context on the test set
Uses the model loaded above to generate answers for the 300 test questions with the retrieved context. Run cells 10 and 11 once for each model.

In [ ]:
run_condition(tok, model, tag, "test", condition="ctx", contexts=CONTEXTS["test"])

qwen3_4b ctx test: 0 done, 300 to go
  25/300 | 1.9 min
  50/300 | 3.9 min
  75/300 | 5.7 min
  100/300 | 7.6 min
  125/300 | 9.4 min
  150/300 | 11.2 min
  175/300 | 13.1 min
  200/300 | 15.1 min
  225/300 | 16.9 min
  250/300 | 18.8 min
  275/300 | 20.7 min
  300/300 | 22.5 min
finished gen_qwen3_4b_ctx_test.jsonl


## 12. D0 summary with context
Prints accuracy, number of errors, how often all 5 samples agree, and D0 AUROC (with 95% confidence interval) for both models on dev and test, for the with-context condition.

In [ ]:
for t in ["qwen3_4b", "phi4_mini"]:
    for s in ["dev", "test"]:
        summarize(t, "ctx", s)

qwen3_4b | ctx | dev: n=150, errors=22, accuracy=0.853
  main unparsed: 0 | all-5-agree: 0.967
  wrong & all 5 repeat same wrong option: 0.864
  D0 AUROC: 0.560  (95% CI 0.492 to 0.640)
qwen3_4b | ctx | test: n=300, errors=36, accuracy=0.880
  main unparsed: 0 | all-5-agree: 0.997
  wrong & all 5 repeat same wrong option: 0.972
  D0 AUROC: 0.514  (95% CI 0.500 to 0.547)
phi4_mini | ctx | dev: n=150, errors=33, accuracy=0.780
  main unparsed: 0 | all-5-agree: 0.8
  wrong & all 5 repeat same wrong option: 0.576
  D0 AUROC: 0.645  (95% CI 0.552 to 0.738)
phi4_mini | ctx | test: n=300, errors=61, accuracy=0.797
  main unparsed: 0 | all-5-agree: 0.807
  wrong & all 5 repeat same wrong option: 0.492
  D0 AUROC: 0.701  (95% CI 0.632 to 0.767)


## 13. D0 and D1 results on the dev set (with context)
Computes D0 and D1 AUROC for the dev set and saves the scores to `scores_<model>_ctx_dev.csv` in Drive.

In [ ]:
for t in ["qwen3_4b", "phi4_mini"]:
    eval_d1(t, "ctx", "dev")

qwen3_4b | ctx | dev: n=150, errors=22
  D0 AUROC 0.560 (95% CI 0.492-0.640) | D1 AUROC 0.701 (95% CI 0.592-0.803)
  D1 mean score: correct=0.0337 | wrong=0.0566
  where D0 is silent (all 5 agree, n=145): errors=19, D1 AUROC 0.694
phi4_mini | ctx | dev: n=150, errors=33
  D0 AUROC 0.645 (95% CI 0.552-0.738) | D1 AUROC 0.543 (95% CI 0.422-0.658)
  D1 mean score: correct=0.1126 | wrong=0.1235
  where D0 is silent (all 5 agree, n=120): errors=19, D1 AUROC 0.486


## 14. D0 and D1 results on the test set (with context)
Computes D0 and D1 AUROC for the test set and saves the scores to `scores_<model>_ctx_test.csv` in Drive.

In [ ]:
for t in ["qwen3_4b", "phi4_mini"]:
    eval_d1(t, "ctx", "test")

qwen3_4b | ctx | test: n=300, errors=36
  D0 AUROC 0.514 (95% CI 0.500-0.547) | D1 AUROC 0.580 (95% CI 0.483-0.671)
  D1 mean score: correct=0.0420 | wrong=0.0549
  where D0 is silent (all 5 agree, n=299): errors=35, D1 AUROC 0.571
phi4_mini | ctx | test: n=300, errors=61
  D0 AUROC 0.701 (95% CI 0.632-0.767) | D1 AUROC 0.629 (95% CI 0.547-0.699)
  D1 mean score: correct=0.1010 | wrong=0.1408
  where D0 is silent (all 5 agree, n=241): errors=30, D1 AUROC 0.573


## 15. Comparison: no context vs with context
Prints accuracy, errors, D0 AUROC and D1 AUROC side by side for the no-context and with-context runs on the test set, for both models.

In [ ]:
def compare(tag, split="test"):
    for cond in ["no_ctx", "ctx"]:
        s = pd.read_csv(PROJECT_DIR / f"scores_{tag}_{cond}_{split}.csv")
        err = s.error.values.astype(int)
        print(f"{tag} | {cond:6s} | accuracy={1 - err.mean():.3f} | errors={err.sum()} "
              f"| D0 AUROC={roc_auc_score(err, s.d0):.3f} | D1 AUROC={roc_auc_score(err, s.d1):.3f}")

for t in ["qwen3_4b", "phi4_mini"]:
    compare(t)

qwen3_4b | no_ctx | accuracy=0.893 | errors=32 | D0 AUROC=0.530 | D1 AUROC=0.594
qwen3_4b | ctx    | accuracy=0.880 | errors=36 | D0 AUROC=0.514 | D1 AUROC=0.580
phi4_mini | no_ctx | accuracy=0.820 | errors=54 | D0 AUROC=0.642 | D1 AUROC=0.655
phi4_mini | ctx    | accuracy=0.797 | errors=61 | D0 AUROC=0.701 | D1 AUROC=0.629


## 16. Summary table of all results
Reads all saved score files and builds one table (model, condition, split, accuracy, D0 and D1 AUROC with 95% confidence intervals). The table is saved as `summary.csv` and is used by the dashboard.

In [ ]:
rows = []
for t in ["qwen3_4b", "phi4_mini"]:
    for c in ["no_ctx", "ctx"]:
        for s in ["dev", "test"]:
            d = pd.read_csv(PROJECT_DIR / f"scores_{t}_{c}_{s}.csv")
            y = d.error.values.astype(int)
            l0, h0 = boot_auc(y, d.d0.values)
            l1, h1 = boot_auc(y, d.d1.values)
            rows.append(dict(model=t, condition=c, split=s, n=len(d), errors=int(y.sum()),
                             accuracy=round(1 - y.mean(), 3),
                             d0_auroc=round(roc_auc_score(y, d.d0), 3), d0_lo=round(l0, 3), d0_hi=round(h0, 3),
                             d1_auroc=round(roc_auc_score(y, d.d1), 3), d1_lo=round(l1, 3), d1_hi=round(h1, 3)))
summary = pd.DataFrame(rows)
summary.to_csv(PROJECT_DIR / "summary.csv", index=False)
summary

,model,condition,split,n,errors,accuracy,d0_auroc,d0_lo,d0_hi,d1_auroc,d1_lo,d1_hi
0,qwen3_4b,no_ctx,dev,150,15,0.900,0.533,0.500,0.611,0.672,0.527,0.795
1,qwen3_4b,no_ctx,test,300,32,0.893,0.530,0.496,0.579,0.594,0.483,0.704
2,qwen3_4b,ctx,dev,150,22,0.853,0.560,0.492,0.640,0.701,0.592,0.803
3,qwen3_4b,ctx,test,300,36,0.880,0.514,0.500,0.547,0.580,0.483,0.671
4,phi4_mini,no_ctx,dev,150,25,0.833,0.741,0.639,0.835,0.689,0.581,0.793
5,phi4_mini,no_ctx,test,300,54,0.820,0.642,0.571,0.706,0.655,0.582,0.724
6,phi4_mini,ctx,dev,150,33,0.780,0.645,0.552,0.738,0.543,0.422,0.658
7,phi4_mini,ctx,test,300,61,0.797,0.701,0.632,0.767,0.629,0.547,0.699


## 17. Paired difference tests
Uses bootstrap resampling on the test set to check (a) whether D1 and D0 differ in AUROC and (b) whether adding context changed the AUROC. If the 95% confidence interval of a difference includes 0, the difference is not statistically clear.

In [ ]:
def paired_boot(y, a, b, n=2000, seed=0):
    rng = np.random.default_rng(seed); diffs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        diffs.append(roc_auc_score(y[i], a[i]) - roc_auc_score(y[i], b[i]))
    return np.percentile(diffs, [2.5, 97.5])

def ctx_effect(tag, score, n=2000, seed=0):
    a = pd.read_csv(PROJECT_DIR / f"scores_{tag}_no_ctx_test.csv").set_index("id")
    b = pd.read_csv(PROJECT_DIR / f"scores_{tag}_ctx_test.csv").set_index("id").loc[a.index]
    ya, yb = a.error.values.astype(int), b.error.values.astype(int)
    sa, sb = a[score].values, b[score].values
    obs = roc_auc_score(yb, sb) - roc_auc_score(ya, sa)
    rng = np.random.default_rng(seed); diffs = []
    for _ in range(n):
        i = rng.integers(0, len(a), len(a))
        if ya[i].min() == ya[i].max() or yb[i].min() == yb[i].max():
            continue
        diffs.append(roc_auc_score(yb[i], sb[i]) - roc_auc_score(ya[i], sa[i]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"{tag} {score.upper()}: ctx - no_ctx AUROC = {obs:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

for t in ["qwen3_4b", "phi4_mini"]:
    for c in ["no_ctx", "ctx"]:
        s = pd.read_csv(PROJECT_DIR / f"scores_{t}_{c}_test.csv")
        y = s.error.values.astype(int)
        lo, hi = paired_boot(y, s.d1.values, s.d0.values)
        print(f"{t} {c}: D1 - D0 AUROC = {roc_auc_score(y, s.d1) - roc_auc_score(y, s.d0):+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")
    for sc in ["d0", "d1"]:
        ctx_effect(t, sc)

qwen3_4b no_ctx: D1 - D0 AUROC = +0.065 (95% CI -0.046 to +0.174)
qwen3_4b ctx: D1 - D0 AUROC = +0.067 (95% CI -0.032 to +0.158)
qwen3_4b D0: ctx - no_ctx AUROC = -0.016 (95% CI -0.073 to +0.034)
qwen3_4b D1: ctx - no_ctx AUROC = -0.014 (95% CI -0.157 to +0.125)
phi4_mini no_ctx: D1 - D0 AUROC = +0.013 (95% CI -0.075 to +0.096)
phi4_mini ctx: D1 - D0 AUROC = -0.072 (95% CI -0.154 to +0.011)
phi4_mini D0: ctx - no_ctx AUROC = +0.059 (95% CI -0.038 to +0.152)
phi4_mini D1: ctx - no_ctx AUROC = -0.026 (95% CI -0.109 to +0.054)
